In [ ]:
import os

# --- Configuration ---
import os, sys, time, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
%matplotlib inline

sys.path.insert(0, str(Path.cwd().parent.parent))
from shared.data_loader import prepare_hierarchical_data_from_splits
from shared.metrics import (compute_all_metrics, compute_per_class_f1,
                             plot_confusion_matrix, plot_per_class_f1)
from shared.utils import set_seed

SEED = 42
N_JOBS = int(os.environ.get("N_JOBS", 26))

os.environ["OMP_NUM_THREADS"] = "1"
set_seed(SEED)

PROJECT_ROOT = Path.cwd().parent.parent
DATASET_DIR = Path("/home/infres/tandriam-25/fil_rouge/dataset")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})


import joblib
print(f"Configuration terminée. N_JOBS={N_JOBS}, backend=threading")

In [ ]:
# --- Chargement des données (avec cache) ---
import joblib as _jl

_cache_path = PROJECT_ROOT / "models" / "baselines" / "_data_cache.pkl"
_cache_path.parent.mkdir(parents=True, exist_ok=True)

if _cache_path.exists():
    print("Chargement depuis le cache...")
    t0 = time.time()
    data = _jl.load(_cache_path)
    print(f"Cache chargé en {time.time() - t0:.0f}s")
else:
    print("Chargement des splits train/val/test...")
    t0 = time.time()
    data = prepare_hierarchical_data_from_splits(
        train_path=str(DATASET_DIR / "train_split.jsonl"),
        val_path=str(DATASET_DIR / "val_split.jsonl"),
        test_path=str(DATASET_DIR / "test_split.jsonl"),
        n_samples=None,
        seed=SEED,
    )
    _jl.dump(data, _cache_path, compress=3)
    print(f"Chargement terminé en {time.time() - t0:.0f}s — cache sauvegardé")

print(f"  Train : {len(data['X_train']):,} | Val : {len(data['X_val']):,} | Test : {len(data['X_test']):,}")
print(f"  L1 : {data['num_classes_l1']} | L2 : {data['num_classes_l2']}")

<a id="section-2"></a>
## 2. Vectorisation TF-IDF

On vectorise les textes avec TF-IDF (50K features, unigrammes + bigrammes, sublinear TF).
Le vectorizer est fit sur le train set uniquement, puis appliqué sur val et test.


In [ ]:
# --- Stopwords ---
import nltk
try:
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))

stopwords_combined = list(french_stops | english_stops)
print(f"Stopwords : {len(stopwords_combined)} mots (FR + EN)")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Vectorisation TF-IDF ---
_save_dir = PROJECT_ROOT / "models" / "baselines" / "tfidf"
_save_dir.mkdir(parents=True, exist_ok=True)
_tfidf_path = _save_dir / "tfidf_vectorizer.joblib"

if _tfidf_path.exists():
    tfidf = joblib.load(_tfidf_path)
    print(f"TfidfVectorizer chargé depuis {_tfidf_path}")
else:
    print("Vectorisation TF-IDF...")
    t0 = time.time()
    tfidf = TfidfVectorizer(
        max_features=50_000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=3,
    )
    tfidf.fit(data["X_train"])
    joblib.dump(tfidf, _tfidf_path)
    print(f"TfidfVectorizer entraîné en {time.time()-t0:.0f}s — sauvegardé")

# Transform parallèle par chunks (threads, pas de copie mémoire)
from scipy.sparse import vstack as sparse_vstack
from joblib import Parallel, delayed

def _parallel_transform(vectorizer, texts, n_jobs):
    chunk_size = max(1, len(texts) // n_jobs)
    chunks = [texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)]
    results = Parallel(n_jobs=n_jobs, backend='threading')(
        delayed(vectorizer.transform)(chunk) for chunk in chunks
    )
    return sparse_vstack(results)

print(f"Transformation des splits ({N_JOBS} threads)...")
t0 = time.time()
X_train_tfidf = _parallel_transform(tfidf, data["X_train"], N_JOBS)
X_val_tfidf = _parallel_transform(tfidf, data["X_val"], N_JOBS)
X_test_tfidf = _parallel_transform(tfidf, data["X_test"], N_JOBS)
print(f"Transformation terminée en {time.time()-t0:.0f}s")
print(f"Shape: train {X_train_tfidf.shape}, val {X_val_tfidf.shape}, test {X_test_tfidf.shape}")

# Libérer les textes bruts (on n'a plus besoin que des matrices sparse + labels)
del data["X_train"], data["X_val"], data["X_test"]
gc.collect()
print("Textes bruts libérés — seules les matrices sparse restent en mémoire")

<a id="section-3"></a>
## 3. Classification L1 (157 catégories)

On utilise LinearSVC avec `class_weight='balanced'` pour compenser le déséquilibre de classes.


In [ ]:
# --- Entraînement L1 ---
# OVR + fork fonctionne pour sparse CSR (CoW sur numpy contiguous = pas de duplication)
_clf_l1_path = PROJECT_ROOT / "models" / "baselines" / "tfidf" / "clf_l1.joblib"

if _clf_l1_path.exists():
    clf_l1 = joblib.load(_clf_l1_path)
    print(f"Classifieur L1 chargé depuis {_clf_l1_path}")
    train_time_l1 = 0
else:
    from sklearn.svm import LinearSVC
    from sklearn.multiclass import OneVsRestClassifier
    print(f"Entraînement L1 : OVR(LinearSVC) — {N_JOBS} workers (sparse CoW)...")
    t0 = time.time()
    clf_l1 = OneVsRestClassifier(
        LinearSVC(C=1.0, class_weight="balanced", max_iter=2000), n_jobs=N_JOBS,
    )
    clf_l1.fit(X_train_tfidf, data["y_l1_train"])
    train_time_l1 = time.time() - t0
    joblib.dump(clf_l1, _clf_l1_path)
    print(f"L1 entraîné en {train_time_l1:.0f}s — sauvegardé")

print("Prédiction L1...")
t0 = time.time()
preds_l1_train = clf_l1.predict(X_train_tfidf)
preds_l1_val = clf_l1.predict(X_val_tfidf)
preds_l1_test = clf_l1.predict(X_test_tfidf)
print(f"Prédiction L1 terminée en {time.time()-t0:.0f}s")

In [ ]:
# --- Top / Bottom classes L1 ---
per_class_f1_l1 = compute_per_class_f1(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"]
)

sorted_f1 = sorted(per_class_f1_l1.items(), key=lambda x: x[1])
print("=== 10 pires classes L1 ===")
for name, score in sorted_f1[:10]:
    print(f"  {name:40s} F1 = {score:.4f}")

print(f"\n=== 10 meilleures classes L1 ===")
for name, score in sorted_f1[-10:]:
    print(f"  {name:40s} F1 = {score:.4f}")


<a id="section-4"></a>
## 4. Classification L2 (6000+ sous-catégories)

Même approche LinearSVC, mais sur les sous-catégories L2. Avec 6000+ classes,
l'entraînement sera plus long et le F1 macro plus bas (beaucoup de classes rares).


In [ ]:
# --- Entraînement L2 ---
# SGDClassifier + fork fonctionne pour sparse CSR (CoW)
_clf_l2_path = PROJECT_ROOT / "models" / "baselines" / "tfidf" / "clf_l2.joblib"

if _clf_l2_path.exists():
    clf_l2 = joblib.load(_clf_l2_path)
    print(f"Classifieur L2 chargé depuis {_clf_l2_path}")
    train_time_l2 = 0
else:
    from sklearn.linear_model import SGDClassifier
    print(f"Entraînement L2 : SGDClassifier — {N_JOBS} workers (sparse CoW)...")
    t0 = time.time()
    clf_l2 = SGDClassifier(
        loss="hinge", class_weight="balanced",
        alpha=1e-4, max_iter=100, n_jobs=N_JOBS,
    )
    clf_l2.fit(X_train_tfidf, data["y_l2_train"])
    train_time_l2 = time.time() - t0
    joblib.dump(clf_l2, _clf_l2_path)
    print(f"L2 entraîné en {train_time_l2:.0f}s — sauvegardé")

print("Prédiction L2...")
t0 = time.time()
preds_l2_test = clf_l2.predict(X_test_tfidf)
print(f"Prédiction L2 terminée en {time.time()-t0:.0f}s")

<a id="section-5"></a>
## 5. Métriques hiérarchiques

Le TF-IDF entraîne L1 et L2 **indépendamment** — il n'y a aucune contrainte hiérarchique.
Le HCR (Hierarchical Consistency Rate) mesure la cohérence des prédictions : est-ce que le L2 prédit
est un enfant valide du L1 prédit ?


In [ ]:
# --- Métriques hiérarchiques ---
all_metrics = compute_all_metrics(
    labels_l1=data["y_l1_test"], preds_l1=preds_l1_test,
    labels_l2=data["y_l2_test"], preds_l2=preds_l2_test,
    l2_to_l1_mapping=data["l2_to_l1_mapping"],
)

print("=== Métriques TF-IDF + LinearSVC/SGD (Test Set) ===")
for k, v in all_metrics.items():
    if isinstance(v, float):
        print(f"  {k:30s} : {v:.4f}")

summary = pd.DataFrame([{
    "Modèle": "TF-IDF + LinearSVC/SGD",
    "F1 L1 macro": all_metrics["f1_l1_macro"],
    "F1 L2 macro": all_metrics["f1_l2_macro"],
    "HCR": all_metrics.get("hcr", 0),
}])
display(summary.style.format({
    "F1 L1 macro": "{:.4f}", "F1 L2 macro": "{:.4f}", "HCR": "{:.4f}",
}).set_caption("TF-IDF Baseline — dataset complet 5.5M"))

<a id="section-6"></a>
## 6. Visualisations


In [ ]:
# --- Matrice de confusion L1 (top 30) ---
fig_cm_l1 = plot_confusion_matrix(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"],
    title="Matrice de confusion L1 — TF-IDF Baseline", top_n=30,
)
plt.show()


In [ ]:
# --- F1 par classe L1 ---
fig_f1_l1 = plot_per_class_f1(
    per_class_f1_l1,
    title="F1 par classe L1 — TF-IDF Baseline",
    top_n=20,
)
plt.show()


In [ ]:
# --- F1 par classe L2 (top/bottom) ---
per_class_f1_l2 = compute_per_class_f1(
    data["y_l2_test"], preds_l2_test, data["l2_encoder"]
)

fig_f1_l2 = plot_per_class_f1(
    per_class_f1_l2,
    title="F1 par classe L2 — TF-IDF Baseline",
    top_n=20,
)
plt.show()

# Stats L2
f1_values_l2 = list(per_class_f1_l2.values())
print(f"F1 L2 — Moyenne: {np.mean(f1_values_l2):.4f}, Médiane: {np.median(f1_values_l2):.4f}")
print(f"  Classes L2 avec F1 = 0 : {sum(1 for v in f1_values_l2 if v == 0)}")
print(f"  Classes L2 avec F1 > 0.5 : {sum(1 for v in f1_values_l2 if v > 0.5)}")


In [ ]:
# --- Sauvegarde per-class F1 en JSON (pour la comparaison) ---
import json as _json
_out_dir = PROJECT_ROOT / "results" / "baselines"
_out_dir.mkdir(parents=True, exist_ok=True)

_l1_path = _out_dir / "per_class_f1_l1_tfidf.json"
_l2_path = _out_dir / "per_class_f1_l2_tfidf.json"
_l1_path.write_text(_json.dumps(per_class_f1_l1, ensure_ascii=False, indent=2))
_l2_path.write_text(_json.dumps(per_class_f1_l2, ensure_ascii=False, indent=2))
print(f"Per-class F1 sauvegardé :")
print(f"  L1: {_l1_path} ({len(per_class_f1_l1)} classes)")
print(f"  L2: {_l2_path} ({len(per_class_f1_l2)} classes)")


<a id="section-8"></a>
## 8. Conclusion

### Forces du TF-IDF
- **Rapidité** : entraînement en quelques minutes (vs ~70h pour HYDRA)
- **Simplicité** : pas de GPU, pas de fine-tuning, peu d'hyperparamètres
- **F1 L1 compétitif** : ~86% macro — une baseline solide

### Limites
- **Pas de compréhension sémantique** : les synonymes et paraphrases ne sont pas capturés
- **Pas de hiérarchie** : L1 et L2 sont prédits indépendamment → HCR faible
- **F1 L2 limité** : trop de classes rares pour un modèle linéaire sans contextualisation

### Ce que HYDRA apporte
- **Embeddings contextuels** (CamemBERT) : compréhension sémantique profonde
- **Architecture multi-tête** : partage d'informations entre L1 et L2
- **Focal Loss** : meilleure gestion du déséquilibre de classes
- **Hiérarchie** : possibilité de contraindre L2 conditionnellement à L1

---
*Notebook généré dans le cadre du projet de classification hiérarchique Rakuten — Télécom Paris x Rakuten France, 2026.*
